# Results of the paper, recomputed

Achimescu & Chachev (2021), *Raising the Flag: Monitoring User Perceived Disinformation on
Reddit*, Information 12(1), 4. https://doi.org/10.3390/info12010004

Each section recomputes one table or set of figures of the paper with the functions of the
`stroo` package.

**Data needed** (none of it is in the repository; paths are relative to `data/`):

| Folder | Files | Source |
|---|---|---|
| `labelled/` | `classified_regex_labeled_train.csv`, `classified_regex_labeled_test.csv` | Figshare, doi:10.6084/m9.figshare.13315259 |
| `dashboard/` | the six `app_*.csv` tables | Figshare, doi:10.6084/m9.figshare.13174136 |
| `factcheck/` | `SSD.csv` | Figshare, doi:10.6084/m9.figshare.13174145 |
| `processed/` | `MATCH_2020_01.csv` ... `MATCH_2020_06.csv` | pipeline output (not public; Table 1 and Table 5 only) |

Sections 1 and 3 parse text and train models, so their figures depend on the versions of
spaCy and scikit-learn installed.

In [1]:
import warnings

import pandas as pd

from stroo import settings
from stroo.validation import factcheck, metrics

warnings.filterwarnings("ignore")
pd.set_option("display.precision", 3)
DATA = settings.DATA_DIR

## 1. Table 1 - frequency of the patterns

Share of pattern matches by pattern group, over the sentences flagged from January to June
2020.

In [2]:
matches = pd.concat(
    pd.read_csv(DATA / "processed" / f"MATCH_2020_{month:02d}.csv", lineterminator="\n")
    for month in range(1, 7)
)
flagged_sentences = matches.loc[matches["flag"] > 0, "sent"].astype(str)
print(f"{len(matches):,} sentences with a flag keyword, {len(flagged_sentences):,} match a pattern")
metrics.pattern_frequencies(flagged_sentences).round(3).rename("share of matches").to_frame()

522,320 sentences with a flag keyword, 96,518 match a pattern


,share of matches
"o, ao",0.324
"svo, svao",0.280
"sva, svna, svna2",0.165
naao,0.136
"vo, vao",0.034
"stvo, stvao",0.030
"ivo, ivao",0.022
"yvo, yvao",0.004
"ivsva, ivsvo, ivsvao",0.005
"svsv, svsav",0.001


## 2. Tables 2 and 3 - manual annotation, keyword filter vs POS matcher

Computed from the annotated comments published on Figshare.

In [3]:
train = metrics.load_labelled(metrics.TRAIN_FILE)
test = metrics.load_labelled(metrics.TEST_FILE)
valid_train, valid_test = metrics.valid(train), metrics.valid(test)

table_2 = pd.DataFrame({
    "Set 1 (Jan-Jul, keyword matches)": [600, (valid_train["sample"] == "all").sum()],
    "Set 2 (Jan-Jul, POS matches)": [600, (valid_train["sample"] == "pos").sum()],
    "Set 3 (Aug-Oct, keyword matches)": [len(test), len(valid_test)],
}, index=["N", "valid N"])
table_2["Total"] = table_2.sum(axis=1)
table_2

,"Set 1 (Jan-Jul, keyword matches)","Set 2 (Jan-Jul, POS matches)","Set 3 (Aug-Oct, keyword matches)",Total
N,600,600,300,1500
valid N,573,591,291,1455


In [4]:
metrics.keyword_vs_pos(train).round(2)

,prec_KEYWORD,precision_POS,recall_POS,F1_POS,n_comments
disinformation,0.29,0.65,0.32,0.43,216
fakenews,0.27,0.73,0.54,0.62,257
bs,0.23,0.52,0.50,0.51,241
misleading,0.83,0.99,0.44,0.61,233
unreliable,0.15,0.67,0.36,0.47,204
propaganda,0.51,0.87,0.48,0.62,248


## 3. Table 4 - POS matcher vs machine-learning baselines

Random forest and logistic regression are trained on the January-July comments and all three
methods are scored on the August-October test set. This cell takes a few minutes.

In [5]:
from stroo.validation import ml_baselines

ml_baselines.compare(n_jobs=4).round(2)

,POS matcher,L1/L2 logistic regression,Random forests
accuracy,0.72,0.66,0.75
precision,0.73,0.57,0.75
recall,0.45,0.52,0.53
F1,0.55,0.54,0.62
AUC-ROC,0.67,0.63,0.71
n_comments,291.00,291.00,291.00


## 4. Section 5 - descriptive statistics

Flagged posts of weeks 1-23 of 2020 in the dashboard table.

In [6]:
posts = pd.read_csv(DATA / "dashboard" / "app_subm_day.csv").query("week <= 23")
print(f"Flagged posts: {len(posts):,}")
print(f"COVID-related: {posts['subm_covid'].mean():.0%}")

times_flagged = posts["flag"].clip(upper=3).value_counts(normalize=True).sort_index()
times_flagged.index = ["once", "twice", "three times or more"]
print("\nTimes flagged:")
print(times_flagged.round(3).to_string())

Flagged posts: 33,609
COVID-related: 47%

Times flagged:
once                   0.826
twice                  0.106
three times or more    0.068


In [7]:
flag_types = ["disinformation", "fakenews", "bs", "misleading", "unreliable", "propaganda"]
by_type = posts[flag_types].sum()
(by_type / by_type.sum()).round(3).sort_values(ascending=False).rename("share of flags").to_frame()

,share of flags
bs,0.366
misleading,0.253
fakenews,0.232
propaganda,0.119
disinformation,0.026
unreliable,0.004


In [8]:
by_category = posts["subreddit_cat"].value_counts(normalize=True).round(3)
by_month = posts.groupby("month").size().rename("flagged posts")
print(by_category.to_string(), "\n")
print(by_month.to_string())

subreddit_cat
coronavirus           0.277
generic_politics      0.256
generic_news          0.181
local                 0.127
ideological           0.091
technology            0.028
science_health        0.025
business_economics    0.013 

month
1.0     4155
2.0     4633
3.0    10137
4.0     7502
5.0     5861
6.0     1321


### Table 5 - volume of data

Keyword matches are counted from the monthly MATCH files. The numbers of posts and comments
need the raw monthly files and are quoted from the paper.

In [9]:
table_5 = pd.DataFrame(
    {"N": [3_395_847, 17_254_621, len(matches), len(posts)]},
    index=["posts (from the paper)", "comments (from the paper)", "keyword matches",
           "POS matches"],
)
table_5["N/month"] = (table_5["N"] / 6).astype(int)
table_5["N/week"] = (table_5["N"] / 23).round(2)
table_5["N/day"] = (table_5["N"] / 182).round(2)
table_5

,N,N/month,N/week,N/day
posts (from the paper),3395847,565974,147645.52,18658.50
comments (from the paper),17254621,2875770,750200.91,94805.61
keyword matches,522320,87053,22709.57,2869.89
POS matches,33609,5601,1461.26,184.66


### Section 5.1 - weekly share of posts flagged (Figure 2)

The weeks with the highest share of flagged posts.

In [10]:
weekly = (
    pd.read_csv(DATA / "dashboard" / "app_subr_week.csv").query("week <= 23")
    .groupby("week")[["flag", "n_subm_kept"]].sum()
)
weekly["percent flagged"] = (100 * weekly["flag"] / weekly["n_subm_kept"]).round(1)
weekly.sort_values("percent flagged", ascending=False).head(5)

,flag,n_subm_kept,percent flagged
week,,,
9.0,1727.0,10309.0,16.8
5.0,1363.0,18006.0,7.6
18.0,1999.0,26872.0,7.4
14.0,2756.0,64647.0,4.3
16.0,2282.0,59915.0,3.8


## 5. Section 5.4 - topics among COVID-19 posts

K-means with three clusters on the five title-embedding components, as in the dashboard's
Clustering tab. Cluster numbers are arbitrary and the solution depends on the random start.

In [11]:
from sklearn.cluster import MiniBatchKMeans

from stroo.dashboard.components import common_words
from stroo.dashboard.data import COMPONENTS

covid = posts[posts["subm_covid"]].copy()
kmeans = MiniBatchKMeans(n_clusters=3, batch_size=1000, random_state=777).fit(covid[COMPONENTS])
covid["cluster"] = kmeans.predict(covid[COMPONENTS]) + 1
for cluster, titles in covid.groupby("cluster")["subm_title"]:
    words = ", ".join(common_words(titles, n_words=9)["word"])
    print(f"CL{cluster} ({len(titles):,} posts): {words}")
share = pd.crosstab(covid["month"].astype(int), covid["cluster"], normalize="index").round(2)
share.loc[[1, 6]]

CL1 (6,733 posts): coronavirus, people, virus, pandemic, cases, patients, health, deaths, study
CL2 (5,241 posts): coronavirus, trump, crisis, response, pandemic, americans, covid-19, president, trump's
CL3 (3,937 posts): coronavirus, china, chinese, wuhan, virus, cases, lockdown, coronavirus:, covid-19


cluster,1,2,3
month,,,
1,0.36,0.14,0.51
6,0.51,0.33,0.16


## 6. Section 5.5 - comparison with a fact-checking website

`SSD.csv` has one row per post shared from January to June 2020; its `fakedomain` column
marks posts that link to a website on NewsGuard's list.

In [12]:
ssd = pd.read_csv(DATA / "factcheck" / "SSD.csv")
result = pd.Series(factcheck.compare_with_suspicious(ssd))
result[[
    "posts from suspicious domains",
    "posts from suspicious domains flagged",
    "share of posts from suspicious domains flagged",
    "share of posts from other domains flagged",
    "share of suspicious domains flagged",
    "share of other domains flagged",
]].rename("value").to_frame()

,value
posts from suspicious domains,23076.000
posts from suspicious domains flagged,830.000
share of posts from suspicious domains flagged,0.036
share of posts from other domains flagged,0.028
share of suspicious domains flagged,0.518
share of other domains flagged,0.071


In [13]:
factcheck.suspicious_domains_per_subreddit(ssd).head(6)

subreddit
conspiracy       71
Coronavirus      62
worldpolitics    50
Conservative     49
DrainTheSwamp    47
news             46
Name: domain, dtype: int64